In [7]:
import pandas as pd
from pathlib import Path

# Definir rutas base
BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DIR_VENTAS = BASE / "data" / "fuente" / "ventas"
DIR_MAESTROS_V1 = BASE / "data" / "fuente" / "maestros" / "v1"
DIR_MAESTROS_V2 = BASE / "data" / "fuente" / "maestros" / "v2"

In [8]:
# 2. Leer y consolidar todos los archivos CSV de ventas diarios en un solo DataFrame
archivos_ventas = list(DIR_VENTAS.glob("ventas_*.csv"))
df_ventas = pd.concat((pd.read_csv(f, dtype=str, keep_default_na=False) for f in archivos_ventas), ignore_index=True)

print(f"Filas antes de limpieza: {len(df_ventas)}")


Filas antes de limpieza: 504749


In [9]:
# 3. Limpieza de Consistencia
# a) SKUs en minúsculas y con espacios
df_ventas["sku"] = df_ventas["sku"].str.strip().str.upper()

# b) Canal con otra grafía
mapeo_canal = {"Tienda fisica": "Tienda física", "ecommerce": "E-commerce", "APP": "App"}
df_ventas["canal"] = df_ventas["canal"].replace(mapeo_canal)

# c) Sucursal vacía
df_ventas.loc[df_ventas["sucursal"] == "", "sucursal"] = "Desconocida"

# d) RUT sin puntos (ej. 12345678-5 -> 12.345.678-5)
def formatear_rut(rut):
    if not rut or "-" not in rut: return rut
    cuerpo, dv = rut.split("-")
    if "." not in cuerpo: # Si no tiene puntos, se los agregamos
        cuerpo = f"{int(cuerpo):,}".replace(",", ".")
    return f"{cuerpo}-{dv}"

df_ventas["rut_cliente"] = df_ventas["rut_cliente"].apply(formatear_rut)

# e) SKU y Campaña inexistente
# Cargamos los maestros reales para validar cuáles existen realmente
productos_validos = pd.read_json(DIR_MAESTROS_V2 / "productos.json")["sku"].unique()
campanas_validas = pd.read_csv(DIR_MAESTROS_V2 / "campanas.csv")["cod_campana"].unique()

df_ventas.loc[~df_ventas["sku"].isin(productos_validos), "sku"] = "DESC-000" # SKU por defecto
df_ventas.loc[~df_ventas["cod_campana"].isin(campanas_validas) & (df_ventas["cod_campana"] != ""), "cod_campana"] = ""


In [10]:
# 4. Limpieza de Validez
# a) Fechas con '/' e imposibles
df_ventas["fecha"] = df_ventas["fecha"].str.replace("/", "-")
df_ventas["fecha"] = pd.to_datetime(df_ventas["fecha"], errors="coerce") # Convierte fechas malas (ej. 2025-02-30) a NaT
df_ventas = df_ventas.dropna(subset=["fecha"]) # Eliminamos las filas con fechas imposibles

# b) Montos no numéricos y negativos
df_ventas["monto"] = pd.to_numeric(df_ventas["monto"], errors="coerce")
df_ventas = df_ventas.dropna(subset=["monto"]) # Eliminamos los montos "N/A"
df_ventas["monto"] = df_ventas["monto"].abs() # Convertimos negativos a positivos absolutos

In [11]:
# 5. Limpieza de Unicidad
# Eliminar líneas duplicadas conservando la primera aparición
df_ventas = df_ventas.drop_duplicates(subset=["id_venta"], keep="first")

print(f"Filas después de limpieza: {len(df_ventas)}")

Filas después de limpieza: 498753


In [12]:
import pandas as pd
import hashlib
from pathlib import Path

# --- DEFINICIÓN DE RUTAS ---
BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DIR_MAESTROS_V1 = BASE / "data" / "fuente" / "maestros" / "v1"
DIR_MAESTROS_V2 = BASE / "data" / "fuente" / "maestros" / "v2"

# ---------------------------------------------------------
# 1. DIM_GEOGRAFIA (Modelo Copo de Nieve)
# ---------------------------------------------------------
# Extraemos comunas y regiones únicas de sucursales y clientes para normalizar[cite: 10]
suc_v2 = pd.read_csv(DIR_MAESTROS_V2 / "sucursales.csv")
cli_v2 = pd.read_csv(DIR_MAESTROS_V2 / "clientes.csv")

geo_suc = suc_v2[['comuna', 'region']].drop_duplicates()
geo_cli = cli_v2[['comuna', 'region']].drop_duplicates()

dim_geografia = pd.concat([geo_suc, geo_cli]).drop_duplicates().reset_index(drop=True)
dim_geografia.insert(0, 'geografia_key', range(1, len(dim_geografia) + 1))

In [13]:
# ---------------------------------------------------------
# 2. DIM_CLIENTE (SCD Tipo 2 y Seudonimización)
# ---------------------------------------------------------
cli_v1 = pd.read_csv(DIR_MAESTROS_V1 / "clientes.csv")

# Función para aplicar SHA-256 al RUT y descartar datos personales (Ley 21.719)[cite: 1, 10]
def hash_rut(rut):
    if pd.isna(rut) or rut == "": return "Desconocido"
    return hashlib.sha256(str(rut).encode('utf-8')).hexdigest()

for df in [cli_v1, cli_v2]:
    df['id_cliente'] = df['rut_cliente'].apply(hash_rut)

# Filtramos solo las columnas analíticas, eliminando nombre, correo y teléfono[cite: 1]
cols_cli = ['id_cliente', 'segmento', 'comuna', 'region']
cli_v1 = cli_v1[cols_cli].drop_duplicates()
cli_v2 = cli_v2[cols_cli].drop_duplicates()

# Aplicar SCD Tipo 2: Historial con fechas de vigencia[cite: 1, 10]
cli_v1['fecha_inicio'] = pd.to_datetime('2019-01-01')
cli_v1['fecha_fin'] = pd.to_datetime('2025-02-28')
cli_v1['es_actual'] = False

cli_v2['fecha_inicio'] = pd.to_datetime('2025-03-01')
cli_v2['fecha_fin'] = pd.to_datetime('2099-12-31')
cli_v2['es_actual'] = True

# Detectar qué clientes realmente cambiaron de segmento o comuna entre v1 y v2[cite: 1]
cambios_cli = cli_v2[~cli_v2.apply(tuple, 1).isin(cli_v1.drop(columns=['fecha_inicio', 'fecha_fin', 'es_actual']).apply(tuple, 1))]['id_cliente']

# Unimos la historia de los que cambiaron con la foto actual de todos
dim_cliente = pd.concat([cli_v1[cli_v1['id_cliente'].isin(cambios_cli)], cli_v2], ignore_index=True)

# Cruzamos con dim_geografia para obtener la 'geografia_key' y eliminamos texto[cite: 10]
dim_cliente = dim_cliente.merge(dim_geografia, on=['comuna', 'region'], how='left')
dim_cliente = dim_cliente.drop(columns=['comuna', 'region'])
dim_cliente.insert(0, 'cliente_key', range(1, len(dim_cliente) + 1))

In [14]:
# ---------------------------------------------------------
# 3. DIM_CAMPANA 
# ---------------------------------------------------------
dim_campana = pd.read_csv(DIR_MAESTROS_V2 / "campanas.csv")
dim_campana.insert(0, 'campana_key', range(1, len(dim_campana) + 1))

# Fila por defecto para ventas que no tienen campaña aplicada[cite: 1, 10]
fila_sin_campana = pd.DataFrame([{
    'campana_key': len(dim_campana) + 1, 'cod_campana': '', 'campana': 'Sin campaña', 
    'tipo_campana': 'N/A', 'pct_descuento': 0.0, 'fecha_inicio': None, 
    'fecha_fin': None, 'categorias_aplica': 'N/A', 'canal_aplica': 'N/A'
}])
dim_campana = pd.concat([dim_campana, fila_sin_campana], ignore_index=True)

In [18]:
# ---------------------------------------------------------
# DIM_SUCURSAL (SCD Tipo 3) y DIM_PRODUCTO (SCD Tipo 2)
# ---------------------------------------------------------
suc_v1 = pd.read_csv(DIR_MAESTROS_V1 / "sucursales.csv")
suc_v2 = pd.read_csv(DIR_MAESTROS_V2 / "sucursales.csv")

# Construir dim_sucursal
dim_sucursal = suc_v2.merge(suc_v1[['cod_sucursal', 'formato']], on='cod_sucursal', suffixes=('', '_anterior'))
dim_sucursal = dim_sucursal.rename(columns={'formato': 'formato_actual'})
dim_sucursal.loc[dim_sucursal['formato_actual'] == dim_sucursal['formato_anterior'], 'formato_anterior'] = None 
dim_sucursal.insert(0, 'sucursal_key', range(1, len(dim_sucursal) + 1))

# Construir dim_producto
prod_v1 = pd.read_json(DIR_MAESTROS_V1 / "productos.json")
prod_v2 = pd.read_json(DIR_MAESTROS_V2 / "productos.json")

prod_v1['fecha_inicio'] = pd.to_datetime('2019-01-01')
prod_v1['fecha_fin'] = pd.to_datetime('2025-02-28')
prod_v1['es_actual'] = False

prod_v2['fecha_inicio'] = pd.to_datetime('2025-03-01')
prod_v2['fecha_fin'] = pd.to_datetime('2099-12-31')
prod_v2['es_actual'] = True

cambios_sku = prod_v2[~prod_v2.apply(tuple,1).isin(prod_v1.drop(columns=['fecha_inicio', 'fecha_fin', 'es_actual']).apply(tuple,1))]['sku']
dim_producto = pd.concat([prod_v1[prod_v1['sku'].isin(cambios_sku)], prod_v2], ignore_index=True)
dim_producto.insert(0, 'producto_key', range(1, len(dim_producto) + 1))

In [19]:
# ---------------------------------------------------------
# 4. DIM_CANAL 
# ---------------------------------------------------------
dim_canal = pd.DataFrame({
    'canal_key': [1, 2, 3],
    'canal': ['Tienda física', 'E-commerce', 'App'],
    'tipo_canal': ['Presencial', 'Digital', 'Digital']
}) #[cite: 10]

print(f"Dim Geografía: {len(dim_geografia)} filas")
print(f"Dim Cliente (SCD2): {len(dim_cliente)} filas (incluye historial)")
print(f"Dim Campaña: {len(dim_campana)} filas")
print(f"Dim Canal: {len(dim_canal)} filas")

Dim Geografía: 15 filas
Dim Cliente (SCD2): 61500 filas (incluye historial)
Dim Campaña: 5 filas
Dim Canal: 3 filas


In [20]:
# 1. Preparar tabla de ventas y generar fecha_key
df_ventas['fecha_key'] = df_ventas['fecha'].dt.strftime('%Y%m%d').astype(int)
df_ventas['cantidad'] = pd.to_numeric(df_ventas['cantidad'], errors='coerce').fillna(1)
df_ventas['rut_hash'] = df_ventas['rut_cliente'].apply(hash_rut) # Usamos la función hash definida antes

In [21]:
# 2. Cruce con dimensiones estáticas (Canal, Sucursal, Campaña)
# Canal
fact = df_ventas.merge(dim_canal[['canal_key', 'canal']], on='canal', how='left')

# Sucursal
fact = fact.merge(dim_sucursal[['sucursal_key', 'cod_sucursal']], left_on='sucursal', right_on='cod_sucursal', how='left')

# Campaña
fact = fact.merge(dim_campana[['campana_key', 'cod_campana', 'pct_descuento']], on='cod_campana', how='left')
fact['pct_descuento'] = fact['pct_descuento'].fillna(0)
# Si no hay campaña, asignamos la clave de "Sin campaña" (la última fila que agregamos)
fact['campana_key'] = fact['campana_key'].fillna(dim_campana['campana_key'].max()) 

In [22]:
# 3. Cruce con dimensiones históricas SCD2 (Producto y Cliente)
# PRODUCTO: Cruzamos por SKU y filtramos por fecha de vigencia
fact = fact.merge(dim_producto[['producto_key', 'sku', 'fecha_inicio', 'fecha_fin', 'precio_lista', 'costo_unitario']], on='sku', how='inner')
fact = fact[(fact['fecha'] >= fact['fecha_inicio']) & (fact['fecha'] <= fact['fecha_fin'])]
fact = fact.drop(columns=['fecha_inicio', 'fecha_fin'])

# CLIENTE: Cruzamos por RUT hasheado y filtramos por fecha de vigencia
fact = fact.merge(dim_cliente[['cliente_key', 'id_cliente', 'fecha_inicio', 'fecha_fin']], left_on='rut_hash', right_on='id_cliente', how='inner')
fact = fact[(fact['fecha'] >= fact['fecha_inicio']) & (fact['fecha'] <= fact['fecha_fin'])]
fact = fact.drop(columns=['fecha_inicio', 'fecha_fin', 'rut_hash', 'id_cliente'])


In [23]:
# 4. Cálculo de Métricas Finales
fact['monto_bruto'] = fact['cantidad'] * fact['precio_lista']
fact['descuento_calculado'] = fact['monto_bruto'] * fact['pct_descuento']
fact['monto_neto'] = fact['monto_bruto'] - fact['descuento_calculado']
fact['costo_total'] = fact['cantidad'] * fact['costo_unitario']
fact['margen'] = fact['monto_neto'] - fact['costo_total']

# Extraer nro_boleta y nro_linea desde id_venta (ej: 'V-1000-1' -> boleta 1000, linea 1)
fact[['nro_boleta', 'nro_linea']] = fact['id_venta'].str.extract(r'V-(\d+)-(\d+)')

# 5. Limpieza final: Seleccionar solo las columnas del modelo dimensional
columnas_finales = [
    'id_venta', 'nro_boleta', 'nro_linea', 
    'fecha_key', 'producto_key', 'cliente_key', 'sucursal_key', 'canal_key', 'campana_key', 
    'cantidad', 'precio_lista', 'monto_bruto', 'descuento_calculado', 'monto_neto', 'costo_total', 'margen'
]
fact_ventas = fact[columnas_finales].drop_duplicates(subset=['id_venta'])

print(f"Fact Ventas generada exitosamente con {len(fact_ventas)} registros.")
print(fact_ventas.head(3))

Fact Ventas generada exitosamente con 456623 registros.
  id_venta nro_boleta nro_linea  fecha_key  producto_key  cliente_key  \
0        1        NaN       NaN   20250101             4        27452   
2        2        NaN       NaN   20250101            16        27452   
4        3        NaN       NaN   20250101            15         3918   

   sucursal_key  canal_key  campana_key  cantidad  precio_lista  monto_bruto  \
0           NaN          1            5         3        349000      1047000   
2           NaN          1            5         1         89000        89000   
4           NaN          1            5         1        499000       499000   

   descuento_calculado  monto_neto  costo_total    margen  
0                  0.0   1047000.0       765000  282000.0  
2                  0.0     89000.0        61000   28000.0  
4                  0.0    499000.0       395000  104000.0  


In [27]:
from sqlalchemy import create_engine
import urllib

# 1. Cambiamos localhost por un punto (.)
server = r'.\SQLEXPRESS' 
database = 'DataRetail360' 
driver = 'ODBC Driver 17 for SQL Server'

In [28]:
# 2. Agregamos TrustServerCertificate=yes a la conexión
params = urllib.parse.quote_plus(f'DRIVER={{{driver}}};SERVER={server};DATABASE={database};Trusted_Connection=yes;TrustServerCertificate=yes;')
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

In [29]:
# 3. Cargar las tablas a la base de datos
print("Iniciando carga a SQL Server...")

try:
    dim_geografia.to_sql('Dim_Geografia', con=engine, if_exists='replace', index=False)
    dim_canal.to_sql('Dim_Canal', con=engine, if_exists='replace', index=False)
    dim_campana.to_sql('Dim_Campana', con=engine, if_exists='replace', index=False)
    dim_sucursal.to_sql('Dim_Sucursal', con=engine, if_exists='replace', index=False)
    dim_producto.to_sql('Dim_Producto', con=engine, if_exists='replace', index=False)
    dim_cliente.to_sql('Dim_Cliente', con=engine, if_exists='replace', index=False)
    print("✓ Dimensiones cargadas exitosamente.")

    fact_ventas.to_sql('Fact_Ventas', con=engine, if_exists='replace', index=False)
    print("✓ Tabla de hechos cargada exitosamente.")
    
    print("\n¡Pipeline Medallion completado!")

except Exception as e:
    print(f"Error durante la carga: {e}")

Iniciando carga a SQL Server...
✓ Dimensiones cargadas exitosamente.
✓ Tabla de hechos cargada exitosamente.

¡Pipeline Medallion completado!
